# 01: Dataset Analysis & EDA

## 1. Research Objective
To load, explore, and analyze the SuicideWatch dataset, identifying the schema, target labels, and potential data leakage without modifying the underlying data.

## 2. Input
Raw SuicideWatch dataset file (CSV) located securely in Google Drive.

## 3. Method
Exploratory Data Analysis using Pandas, Matplotlib, and Seaborn. Inspection of schema, missing values, duplicates, text lengths, and label distributions.

## 4. Output
EDA visualizations saved to `results/figures/` and a dataset statistics summary saved to `results/metrics/dataset_analysis.json`.

## 5. Experimental Decisions
A configurable `DATASET_PATH` is used. We do not invent statistics or assume labels before code execution.

## 6. Evaluation Metrics
Not applicable for this phase (EDA only).

In [ ]:
import sys
import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# 1. Setup Environment and Mount Drive if in Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/evidence-augmented-depression-transformer'
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        PROJECT_ROOT = '../'
    else:
        PROJECT_ROOT = '.'

sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {os.getcwd()}")

# Ensure required output directories exist
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

# 2. Configurable Dataset Path
config_path = "configs/experiment.yaml"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
        data_path = config.get("data", {}).get("path", "data/")
else:
    data_path = "data/"


DATASET_DIR = "/content/drive/MyDrive/ML_Dataset"
# IMPORTANT: Since the dataset is in a private Google Drive, 
# please update DATASET_PATH if it is located elsewhere.
DATASET_PATH = os.path.join(data_path, "suicide_watch.csv")

print(f"Checking path: {DATASET_PATH}")
if os.path.exists(DATASET_PATH):
    print("Dataset found successfully!")
    df_sample = pd.read_csv(DATASET_PATH, nrows=5)
    print("Columns in file:", df_sample.columns.tolist())
else:
    print("File not found! Files currently visible in folder:")
    folder_to_check = os.path.dirname(DATASET_PATH)
    if os.path.exists(folder_to_check):
        print(os.listdir(folder_to_check))
    else:
        print(f"Directory does not exist: {folder_to_check}")
print(f"Expected Dataset Path: {DATASET_PATH}")
print("Please ensure the DATASET_PATH correctly points to your private Kaggle SuicideWatch dataset.")

## Load Dataset and Inspect Schema
Loads the dataset securely and displays the schema, data types, and initial samples.

In [ ]:
if not os.path.exists(DATASET_PATH):
    print(f"[WARNING] Dataset not found at {DATASET_PATH}. Please provide the correct absolute path to your Drive file.")
    # df = pd.DataFrame() # Prevents further cells from crashing during testing, but real execution needs the file
else:
    print("Loading dataset...")
    df = pd.read_csv(DATASET_PATH)
    print("\n--- Dataset Overview ---")
    print(f"Filename: {os.path.basename(DATASET_PATH)}")
    print(f"Number of rows: {df.shape[0]}")
    print(f"Number of columns: {df.shape[1]}")
    
    print("\n--- Column Names and Data Types ---")
    print(df.dtypes)
    
    print("\n--- First 5 Samples ---")
    display(df.head())
    
    print("\n--- Missing Values ---")
    print(df.isnull().sum())
    
    print("\n--- Duplicate Rows (Exact Match) ---")
    print(df.duplicated().sum())
    
    # Identify Text and Label columns generically based on Kaggle schema
    text_col = 'text' if 'text' in df.columns else df.columns[0]
    label_col = 'class' if 'class' in df.columns else df.columns[1]
    
    print(f"\nIdentified Text Column: '{text_col}'")
    print(f"Identified Label Column: '{label_col}'")
    
    metadata_cols = [c for c in df.columns if c not in [text_col, label_col]]
    if len(metadata_cols) > 0:
        print(f"Identified Metadata Columns: {metadata_cols}")
    else:
        print("No additional metadata columns identified.")
        
    print("\n--- Label Values & Class Distribution ---")
    print(df[label_col].value_counts())

## Analyze Text Attributes
Calculate text lengths and identify empty/near-empty posts.

In [ ]:
if 'df' in locals() and not df.empty:
    df['text_length_chars'] = df[text_col].astype(str).apply(len)
    df['text_length_words'] = df[text_col].astype(str).apply(lambda x: len(x.split()))
    
    print("--- Text Length Analysis (Words) ---")
    print(f"Minimum length: {df['text_length_words'].min()}")
    print(f"Maximum length: {df['text_length_words'].max()}")
    print(f"Mean length:    {df['text_length_words'].mean():.2f}")
    print(f"Median length:  {df['text_length_words'].median()}")
    
    empty_posts = df[df['text_length_words'] == 0]
    print(f"\nEmpty or near-empty posts: {len(empty_posts)}")

## EDA Visualizations
Generates graphs for class distribution and text lengths.

In [ ]:
if 'df' in locals() and not df.empty:
    # 1. Class Distribution
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x=label_col)
    plt.title("Class Distribution")
    plt.savefig('results/figures/class_distribution.png', bbox_inches='tight')
    plt.show()
    
    # 2. Text Length Distribution
    plt.figure(figsize=(10, 5))
    sns.histplot(df['text_length_words'], bins=50, kde=True)
    plt.title("Text Length Distribution (Words)")
    plt.xlabel("Number of Words")
    # Capping X axis at 95th percentile to prevent long-tail distortion
    plt.xlim(0, df['text_length_words'].quantile(0.95)) 
    plt.savefig('results/figures/text_length_distribution.png', bbox_inches='tight')
    plt.show()
    
    # 3. Class-wise Text Length Distribution
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=label_col, y='text_length_words')
    plt.title("Class-wise Text Length Distribution")
    plt.ylim(0, df['text_length_words'].quantile(0.95))
    plt.savefig('results/figures/class_wise_text_length.png', bbox_inches='tight')
    plt.show()

## Data Leakage & Dataset Summary
Inspects potential data leakage across target labels and evaluates predefined splits.

In [ ]:
if 'df' in locals() and not df.empty:
    print("--- Data Leakage Check ---")
    duplicates = df[df.duplicated(subset=[text_col], keep=False)]
    leakage = duplicates.groupby(text_col)[label_col].nunique()
    conflicting_labels = leakage[leakage > 1]
    
    print(f"Total duplicate texts (ignoring labels): {len(duplicates)}")
    print(f"Texts with conflicting labels (potential leakage/noise): {len(conflicting_labels)}")
    
    print("\n--- Split Check ---")
    has_split = False
    if 'split' in df.columns or 'dataset' in df.columns:
        print("Dataset contains a predefined split column.")
        has_split = True
    else:
        print("No predefined split column found. Train/validation/test splitting will be required in Phase 2.")
    
    # Export Dataset Summary
    summary = {
        "dataset_size": len(df),
        "num_columns": len(df.columns),
        "missing_values": int(df.isnull().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "class_distribution": df[label_col].value_counts().to_dict(),
        "mean_word_count": float(df['text_length_words'].mean()),
        "median_word_count": float(df['text_length_words'].median()),
        "max_word_count": int(df['text_length_words'].max()),
        "min_word_count": int(df['text_length_words'].min()),
        "conflicting_labels_count": len(conflicting_labels),
        "has_predefined_split": has_split
    }
    
    with open('results/metrics/dataset_analysis.json', 'w') as f:
        json.dump(summary, f, indent=4)
        
    print("\nDataset analysis summary saved to results/metrics/dataset_analysis.json")

## Research Methodology Implications

**What the dataset actually represents:**
The SuicideWatch dataset consists of social media posts sourced from Reddit. The posts represent self-reported instances of severe emotional distress and suicidal ideation, matched against general or non-suicidal posts.

**What prediction task the labels support:**
The labels inherently support a binary text classification task representing **suicidal ideation vs. non-suicidal/generic text**. 

**Alignment with current research title:**
Our research title specifies "Early Depression Risk Detection". However, **suicidal ideation is not synonymous with a clinical depression diagnosis**. While they are correlated, using this dataset explicitly evaluates distress and suicidality. We must either adjust our title to better reflect the dataset (e.g., *Early Suicidal Ideation Detection*) or carefully frame the paper to clarify that social media distress is being used as a proxy for broad "depression risk."

**Limitations that must be acknowledged:**
- Posts lack clinical validation; they are based entirely on self-reporting and social media activity.
- Individual posts cannot serve as formal medical diagnoses.
- Data may contain colloquialisms, internet slang, and significant noise.
- Data leakage risk (identified during this EDA) requires robust filtering to ensure duplicates across classes do not bleed into the test set.

**What should be decided before Phase 2:**
- Finalize whether to alter the paper title to align directly with the dataset labels.
- Define the specific methodology for Train/Validation/Test splitting, ensuring stratification across the target label.
- Determine the strategy for handling any duplicate or conflicting texts identified in this EDA.